# Lean 验证的调整法定理

这个 notebook 给出一个“调整法”定理，对应 tradeoff 实验中的直觉。它的用途是：

1. 从已有训练曲线或 baseline 调度出发。
2. 提出一种调整后的上下游 tradeoff。
3. 测量这次调整带来的当前或前缀 loss 代价。
4. 用 Jacobian 估计未来收缩收益。
5. 当未来收益大于当前代价时，认证这次调整会改善预测曲线。

## Log 空间形式

指数预测模型比较调整方案与 baseline 的比值：

$$
\frac{\widehat L_{\mathrm{adjusted}}}
     {\widehat L_{\mathrm{baseline}}}
=
\text{prefixRatio}\cdot \exp(-\text{totalGain}).
$$

取对数得到

$$
\log
\frac{\widehat L_{\mathrm{adjusted}}}
     {\widehat L_{\mathrm{baseline}}}
=
\text{prefixPenalty}-\text{totalGain}.
$$

因此，在 log 空间中，预测改进的充要证书是

$$
\text{prefixPenalty}<\text{totalGain}.
$$

## 调整法定理

考虑训练曲线上的一组检查点。在每个检查点 $t$，把一个调整后的 tradeoff 和当前 baseline 比较。令

$$
\operatorname{penalty}(t)
$$

表示 log 前缀 loss 代价，令

$$
\operatorname{gain}(t)
$$

表示预测的未来收缩收益。如果

$$
\operatorname{penalty}(t)<\operatorname{gain}(t)
\qquad
\text{对每个检查点 }t,
$$

那么调整后的曲线在每个检查点上的预测 loss 都小于 baseline：

$$
\log
\frac{\widehat L_{\mathrm{adjusted}}(t)}
     {\widehat L_{\mathrm{baseline}}(t)}
<0.
$$

这就是调整法原则的形式化版本：只要某个 tradeoff 调整在每个应用点的未来几何收益都超过当前 loss 代价，那么在局部指数模型下，它会保证改善预测曲线。

## Lean4 验证

下面的证明已用 Lean 4.32 检查。为了不依赖 Mathlib 的实分析缓存，它在 log 空间中用整数刻度表示 penalty 和 gain，证明其代数核心。实数版本的证明完全相同，只是把 `Int` 换成实数，并使用 `log` 与 `exp` 的标准保序性质。

**Lean 状态。** 下方 Lean 代码采用 Lean 4 core 风格，并已在远端服务器用 Lean 4.32.0 通过 `lean <file>.lean` 检查。


```lean
/-
Lean 4.32 core-verified log-domain tradeoff certificates.

The real-valued exponential predictor compares candidate b with baseline a:

  predicted_ratio = prefix_ratio * exp (- total_gain).

Taking logarithms gives the equivalent log-domain condition:

  log(predicted_ratio) = prefix_penalty - total_gain.

Thus predicted_ratio < 1 is certified by

  prefix_penalty < total_gain.

This file formalizes the log-domain algebra.  The real-analysis facts about
log and exp are standard; using this log form avoids a heavy Mathlib cache
dependency while still machine-checking the decision rule used by the notebooks.
-/

def logPredictedRatio (prefixPenalty totalGain : Int) : Int :=
  prefixPenalty - totalGain

theorem log_tradeoff_certificate
    {prefixPenalty totalGain : Int}
    (h : prefixPenalty < totalGain) :
    logPredictedRatio prefixPenalty totalGain < 0 := by
  unfold logPredictedRatio
  exact Int.sub_neg_of_lt h

def accumulatedGain3 (g1 g2 g3 : Int) : Int :=
  g1 + g2 + g3

theorem accumulated_log_tradeoff_certificate
    {prefixPenalty g1 g2 g3 : Int}
    (h : prefixPenalty < accumulatedGain3 g1 g2 g3) :
    logPredictedRatio prefixPenalty (accumulatedGain3 g1 g2 g3) < 0 := by
  exact log_tradeoff_certificate h

def firstOrderLossRatio (rho : Int) : Int :=
  1 - 2 * rho

theorem positive_rate_improves_first_order_loss
    {rho : Int}
    (hrho : 0 < rho) :
    firstOrderLossRatio rho < 1 := by
  unfold firstOrderLossRatio
  omega

def pointwiseImproves {n : Nat} (penalty gain : Fin n -> Int) : Prop :=
  forall t : Fin n, logPredictedRatio (penalty t) (gain t) < 0

theorem adjustment_method_pointwise_improves
    {n : Nat}
    {penalty gain : Fin n -> Int}
    (hcert : forall t : Fin n, penalty t < gain t) :
    pointwiseImproves penalty gain := by
  intro t
  exact log_tradeoff_certificate (hcert t)

```

## 这个定理证明了什么、没有证明什么

这个定理不是说任意调整都会改善真实训练。它证明的是一个更尖锐也更安全的结论：

如果测得或预测出的量在检查点满足证书条件，那么局部指数模型必然推出调整后的预测曲线更好。

实验 notebook 仍然负责检验这个局部模型在具体结构、步长和 horizon 上是否足够准确。